# From bits to qubits
**Quantum Computing · Week 1 (Qubits and measurement) · Day 1**

Lesson page: https://sara-aminpour.com/tutorials/qc/w1d1.html

In [ ]:
# On Google Colab, this installs qiskit (about a minute). On your own computer, skip it and
# install the packages once instead: pip install qiskit==2.5.2 scikit-learn torch matplotlib
import sys
if "google.colab" in sys.modules:
    %pip install --quiet qiskit==2.5.2

A qubit state is just two numbers, the **amplitudes**. Square them to get probabilities.
Today: amplitudes with NumPy first, then the same thing with Qiskit.

In [ ]:
import numpy as np

psi = np.array([np.sqrt(0.7), np.sqrt(0.3)])   # (alpha, beta)
probs = np.abs(psi) ** 2                       # Born rule: probability = |amplitude|^2

print("amplitudes   :", psi.round(3))
print("probabilities:", probs.round(3))
print("sum          :", probs.sum().round(3))

A valid state must have probabilities that add up to 1 (it is **normalized**).
Amplitudes may be negative or even complex (`1j` is Python's imaginary unit).

In [ ]:
def is_valid(state):
    return np.isclose(np.sum(np.abs(state) ** 2), 1.0)

candidates = {
    "(1, 0)": np.array([1, 0]),
    "(0.6, 0.8)": np.array([0.6, 0.8]),
    "(0.6, -0.8)": np.array([0.6, -0.8]),
    "(0.5, 0.5)": np.array([0.5, 0.5]),
    "(1, i)/sqrt(2)": np.array([1, 1j]) / np.sqrt(2),
}
for name, state in candidates.items():
    print(f"{name:>15}  ->  {'valid' if is_valid(state) else 'not normalized'}")

## The same thing in Qiskit
`Statevector` holds the amplitudes and knows how to turn them into probabilities
and into simulated measurement results.

In [ ]:
from qiskit.quantum_info import Statevector

sv = Statevector([np.sqrt(0.7), np.sqrt(0.3)])
print(sv)
print("probabilities:", sv.probabilities().round(3))   # [P(0), P(1)]

sv.seed(7)                                  # fixed seed so the "random" result repeats
print("10 measurements:", sv.sample_memory(10))

Each measurement gives just one bit. Repeat it many times and the histogram
approaches the probabilities 0.7 and 0.3.

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(9.5, 3.2), sharey=True, layout="constrained")
for ax, shots in zip(axes, [10, 100, 10000]):
    sv.seed(7)
    counts = sv.sample_counts(shots)
    freq = [counts.get("0", 0) / shots, counts.get("1", 0) / shots]
    ax.bar(["0", "1"], freq, color=["#5577f5", "#ff8a50"], width=0.6)
    ax.axhline(0.7, color="#5577f5", ls="--", lw=1)
    ax.axhline(0.3, color="#ff8a50", ls="--", lw=1)
    ax.set_title(f"{shots} shots")
axes[0].set_ylabel("fraction of shots")
plt.show()

## Your turn: make any vector a qubit
Divide a vector by its length to make it normalized. Fill in `normalize`, then check the three vectors.

In [ ]:
def normalize(v):
    v = np.asarray(v, dtype=complex)
    return v   # TODO: divide v by its length, np.linalg.norm(v)

for v in [[3, 4], [1, 1], [1, 1j]]:
    state = normalize(v)
    print(v, "->", state.round(3), " valid?", is_valid(state))

---
## Solution
Try the exercise yourself first: it's fine to get stuck for a minute. Then compare with this one (there are many good answers).

In [ ]:
def normalize(v):
    v = np.asarray(v, dtype=complex)
    return v / np.linalg.norm(v)

for v in [[3, 4], [1, 1], [1, 1j]]:
    state = normalize(v)
    probs = np.abs(state) ** 2
    print(f"{str(v):>8} -> amplitudes {state.round(3)}  probabilities {probs.round(2)}  valid? {is_valid(state)}")
print("(1, 1) and (1, i) give the same 50/50 odds, yet they are different states. Tomorrow's lesson is about that difference.")